In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
from tools import bar, line
from pathlib import Path

FIGURES_DIR = Path(r'..\reports\figures\compta')
FIG_WIDTH, FIG_HEIGHT = 1600, 500

In [3]:
df = pd.read_csv(r"..\data\processed\2BLONDEL_consolide.csv", sep=';', decimal=',', parse_dates=['date_écriture'])

Preparation des DF

In [4]:
# DF Loyers
loyers = df.query("code_compte == '70600000'")
# Colonne "canal"
conds = [
    loyers['libellé_écriture'].str.contains('BNB|FACTURE HOTE', case=False),
    loyers['libellé_écriture'].str.contains('BOOKING', case=False),
    ]
choices = ['AIRBNB', 'BOOKING']
loyers['canal'] = np.select(conds, choices, default='DIRECT')

In [5]:
# DF Charges
charges = df[df['code_compte'].str.startswith('6') & (df['débit_euro'] > 0)]
# Colonne "categorie"
fixes = ['61610000', '66116000', '62620000', '62780000', '63512000', '62810000', '68112000']
semi_fixes = ['60630000', '61520000', '61560000']
variables = ['60611000', '60614000', '62220000', '62260000', '62270000', '62310000', '62340000', '62510000', '62610000', '65800000']

conds = [
    charges['code_compte'].isin(fixes),
    charges['code_compte'].isin(semi_fixes),
    charges['code_compte'].isin(variables)
]

choices = ['fixe', 'semi_fixe', 'variable']

charges['categorie'] = np.select(conds, choices, default='autre')

In [6]:
# DF Cashflow
recette = loyers.groupby(loyers['date_écriture'].dt.to_period('M'))[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
recette = recette.drop(columns=['crédit_euro', 'débit_euro'])
depense = charges.groupby(charges['date_écriture'].dt.to_period('M'))['débit_euro'].sum().reset_index()
depense = depense.rename(columns={'débit_euro':'charges'})

# Merge
cashflow = recette.merge(depense, how='outer', on='date_écriture')
cashflow = cashflow.fillna(0)

# Cashflow
cashflow['cashflow'] = cashflow['ca_net'] - cashflow['charges']

---
# CA

## Evolution du CA

In [7]:
ca = loyers.groupby(loyers['date_écriture'].dt.to_period('M'))[['crédit_euro', 'débit_euro']].sum()
ca = ca.eval('ca_net = crédit_euro - débit_euro').reset_index()
ca['periode'] = ca['date_écriture'].astype(str)

fig = line(ca, x='periode', y='ca_net', titre='Evolution du CA', palette=px.colors.qualitative.D3)
fig.update_traces(fill='tozeroy', fillcolor='rgba(33, 150, 243, 0.15)')
fig.update_layout(yaxis=dict(title='Chiffre d\'affaire'))

fig.show()
fig.write_image(FIGURES_DIR / "evolution_ca.png", width=FIG_WIDTH, height=FIG_HEIGHT)

## CA Mensuel 2024 vs 2025

In [8]:
ca_mensuel = loyers.groupby([loyers['date_écriture'].dt.month, loyers['année']])[['crédit_euro', 'débit_euro']].sum()
ca_mensuel = ca_mensuel.eval('ca_net = crédit_euro - débit_euro').reset_index()
ca_mensuel = ca_mensuel.sort_values(['année', 'date_écriture'])

mois_labels = {1: 'Jan', 2: 'Fév', 3: 'Mar', 4: 'Avr', 5: 'Mai', 6: 'Jun',
               7: 'Jul', 8: 'Aoû', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Déc'}

ca_mensuel['mois_label'] = ca_mensuel['date_écriture'].map(mois_labels)

fig = bar(ca_mensuel, x='mois_label', y='ca_net', color='année', titre='CA 2024 vs 2025', palette=px.colors.qualitative.D3)
fig.update_layout(yaxis=dict(title='Chiffre d\'affaire'))

fig.show()
fig.write_image(FIGURES_DIR / "ca2024vs2025.png", width=FIG_WIDTH, height=FIG_HEIGHT)

## CA par Plateforme

In [9]:
ca_ptf = loyers.groupby([loyers['date_écriture'].dt.to_period('M'), 'canal'])[['crédit_euro', 'débit_euro']].sum()
ca_ptf = ca_ptf.eval('ca_net = crédit_euro - débit_euro').reset_index()
ca_ptf['periode'] = ca_ptf['date_écriture'].astype(str)

fig = bar(ca_ptf, x='periode', y='ca_net', color='canal', text_auto=False, titre='CA par Plateforme', palette=px.colors.qualitative.D3, barmode='stack')
fig.update_layout(yaxis=dict(title='Chiffre d\'affaire'))

fig.show()
fig.write_image(FIGURES_DIR / "ca_par_ptf.png", width=FIG_WIDTH, height=FIG_HEIGHT)

---
# Charges

## Charges par Categories

In [10]:
categ = charges.groupby(['code_compte', 'intitulé_compte'])['débit_euro'].sum().reset_index()
categ = categ.sort_values('débit_euro')

fig = bar(categ, x='intitulé_compte', y='débit_euro', titre='Charges par Catégories', horizontal=True, palette=px.colors.qualitative.D3)
fig.update_layout(xaxis=dict(ticksuffix=' €'))

fig.show()
fig.write_image(FIGURES_DIR / "charges_par_categ.png", width=FIG_WIDTH, height=FIG_HEIGHT)

## Fixe, Semi-Fixe, Variable

In [11]:
fsv = charges.groupby([charges['date_écriture'].dt.to_period('M'), charges['categorie']])['débit_euro'].sum().reset_index()
fsv['periode'] = fsv['date_écriture'].astype(str)

fig = bar(fsv, x='periode', y='débit_euro', color='categorie', text_auto=False, titre='Charges par Type', palette=px.colors.qualitative.D3)
fig.update_layout(yaxis=dict(ticksuffix=' €'))

fig.show()
fig.write_image(FIGURES_DIR / "charges_par_type.png", width=FIG_WIDTH, height=FIG_HEIGHT)

### Filtres des mois atypiques (Mai 2024 et Decembre 2024)

In [12]:
fsv_filtre = charges[~charges['date_écriture'].dt.to_period('M').isin([pd.Period('2024-05', 'M'), pd.Period('2024-12', 'M')])]
fsv_filtre = fsv_filtre.groupby([fsv_filtre['date_écriture'].dt.to_period('M'), fsv_filtre['categorie']])['débit_euro'].sum().reset_index()
fsv_filtre['periode'] = fsv_filtre['date_écriture'].astype(str)

fig = bar(fsv_filtre, x='periode', y='débit_euro', color='categorie', text_auto=False, titre='Charges par Type — hors mois atypiques (mai et décembre 2024)', palette=px.colors.qualitative.D3)
fig.update_layout(yaxis=dict(ticksuffix=' €'))

fig.show()
fig.write_image(FIGURES_DIR / "charges_par_type_filtre.png", width=FIG_WIDTH, height=FIG_HEIGHT)

---
# Cashflow

In [13]:
couleurs = ['green' if x > 0 else 'red' for x in cashflow['cashflow']]
cashflow['periode'] = cashflow['date_écriture'].astype(str)

fig = bar(cashflow, x='periode', y='cashflow', titre='Cashflow', palette=couleurs)
fig.update_layout(yaxis=dict(ticksuffix=' €'))

fig.show()
fig.write_image(FIGURES_DIR / "cashflow.png", width=FIG_WIDTH, height=FIG_HEIGHT)

### Filtres des mois atypiques (Mai 2024 et Decembre 2024)

In [14]:
cashflow_filtre = cashflow[~cashflow['date_écriture'].isin([pd.Period('2024-05', 'M'), pd.Period('2024-12', 'M')])]
couleurs = ['green' if x > 0 else 'red' for x in cashflow_filtre['cashflow']]
cashflow_filtre['periode'] = cashflow_filtre['date_écriture'].astype(str)

fig = bar(cashflow_filtre, x='periode', y='cashflow', titre='Cashflow — hors mois atypiques (mai et décembre 2024)', palette=couleurs)
fig.update_layout(yaxis=dict(ticksuffix=' €'))

fig.show()
fig.write_image(FIGURES_DIR / "cashflow_filtre.png", width=FIG_WIDTH, height=FIG_HEIGHT)